# In Silico `Toxicity` prediction for `Bacillus Anthracis` Antibiotic Candidates

This workflow uses `admet_ai` predictions to perform a **rule-based safety triage** of candidate molecules.

The goal is to support **early prioritization**, not to make definitive toxicity claims.

We separate predictions into three categories:

- **Core toxicity risk**: direct safety concerns such as hERG, DILI, AMES, and ClinTox
- **Mechanistic alerts**: stress-response and nuclear-receptor pathway signals
- **ADME / metabolism liabilities**: permeability, transporter, and CYP-related liabilities that may affect developability but do not necessarily imply intrinsic toxicity

In [3]:
import sys
import os

# Completely suppress stderr output
sys.stderr = open(os.devnull, 'w')

# Now import everything
import warnings
warnings.filterwarnings('ignore')

In [4]:
import numpy as np
import pandas as pd
import pubchempy as pcp
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from admet_ai import ADMETModel
import matplotlib.pyplot as plt

In [18]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
modelBuildingDataDir = os.path.join(dataDir, 'modelBuildingData/')
resultsDir = os.path.join(dataDir, 'Results/')
saveDir = os.path.join(resultsDir, "Bacillus_anthracis_bestMACAW/")
DORANETmoleculesDataDir = os.path.join(dataDir, 'DORAnet_molecules/')
ARTresultsDir = os.path.join(dataDir, 'Results/Bacillus_anthracis_bestMACAW/')
os.makedirs(saveDir, exist_ok=True)

## Check toxicty for `buyable` AntiviralsData set

In [19]:
BacillusAnthracisVirus_wART_Buyable_predicted_all = pd.read_csv(ARTresultsDir + 'BacillusAnthracis_all_Buyable_AntiVirals_wARTprediction.csv')
max_val = BacillusAnthracisVirus_wART_Buyable_predicted_all['pPotency_prediction'].max()
min_val = BacillusAnthracisVirus_wART_Buyable_predicted_all['pPotency_prediction'].min()
print(f"pPotency_prediction range: {min_val:.3f} → {max_val:.3f}")
BacillusAnthracisVirus_wART_Buyable_predicted_all

pPotency_prediction range: 0.883 → 6.329


,SMILES,pPotency_prediction,pPotency_std,pPotency_lower_95CI,pPotency_upper_95CI,IC50(M)_prediction,IC50(M)_lower_95CI,IC50(M)_upper_95CI
0,Cc1nnc(COC(=O)c2ccc(N3CC(C)CC(C)C3)c([N+](=O)[...,2.420812,1.929846,-1.361687,6.203311,0.003795,6.261649e-07,22.997815
1,Cc1nc2nc(SCCCOC(=O)NC(N)=O)nn2c(C)c1C,3.495472,1.932781,-0.292779,7.283722,0.000320,5.203287e-08,1.962362
2,Cc1ccc(S(=O)(=O)NCC(=O)OCC(=O)N2CC(C)OC(C)C2)cc1,2.896828,1.930401,-0.886757,6.680413,0.001268,2.087308e-07,7.704731
3,CC(C)n1cnc(S(=O)(=O)Nc2ccn(Cc3ccccn3)n2)c1,3.727565,1.934292,-0.063648,7.518778,0.000187,3.028464e-08,1.157837
4,O=C(O)Cc1ccc(S(=O)(=O)Nc2ccc(Cl)cc2O)cc1,4.094989,1.938103,0.296307,7.893671,0.000080,1.277407e-08,0.505467
...,...,...,...,...,...,...,...,...
59746,Nc1nc(Nc2ccccc2F)nc2ccccc12,4.226625,1.954812,0.395194,8.058056,0.000059,8.748705e-09,0.402537
59747,CNc1nc(Nc2ccccc2F)nc2ccccc12,3.635805,1.958712,-0.203271,7.474882,0.000231,3.350567e-08,1.596876
59748,Nc1nc(Nc2ccccc2Cl)nc2ccccc12,4.236636,1.962426,0.390281,8.082991,0.000058,8.260558e-09,0.407117
59749,CNc1nc(Nc2ccccc2Cl)nc2ccccc12,3.757461,1.971891,-0.107446,7.622368,0.000175,2.385790e-08,1.280697


### Read predicted ADMET properties for all candidate SMILES

In [20]:
BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity = pd.read_csv(ARTresultsDir + 'BacillusAnthracis_all_Buyable_AntiVirals_wARTprediction_wToxicity.csv')
max_val = BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity['pPotency_prediction'].max()
min_val = BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity['pPotency_prediction'].min()
print(f"pPotency_prediction range: {min_val:.3f} → {max_val:.3f}")
BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity

FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/Bacillus_anthracis_bestMACAW/BacillusAnthracis_all_Buyable_AntiVirals_wARTprediction_wToxicity.csv'

### Retain potency predictions along with toxicity, mechanistic, and ADME-related endpoints

In [ ]:
# Potency-related columns
potencyCols = [
    "SMILES",
    "pPotency_prediction", "pPotency_std",
    "pPotency_lower_95CI", "pPotency_upper_95CI",
    "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI"
]

# Core toxicity endpoints
coreToxicityCols = [
    "AMES",
    "hERG",
    "DILI",
    "ClinTox",
    "Carcinogens_Lagunin",
    "Skin_Reaction",
    "LD50_Zhu"
]

# Mechanistic alert endpoints
mechanisticAlertCols = [
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
    "NR-AR-LBD",
    "NR-AR",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER-LBD",
    "NR-ER",
    "NR-PPAR-gamma"
]

# Mechanistic alert endpoints
# ADME / metabolism liability endpoints
admeLiabilityCols = [
    "BBB_Martins",
    "Pgp_Broccatelli",
    "Caco2_Wang",
    "HIA_Hou",
    "PAMPA_NCATS",
    "Bioavailability_Ma",
    "CYP1A2_Veith",
    "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels",
    "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels",
    "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels",
    "CYP3A4_Veith"
]

# Keep only columns that exist
selectedCols = potencyCols + [
    col for col in (coreToxicityCols + mechanisticAlertCols + admeLiabilityCols)
    if col in BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity.columns
]

BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity = BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity[selectedCols].copy()

print("Filtered dataframe shape:", BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity.shape)
BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity.head()

### Read ADMET endpoints of the FDA approved Drug data from `Drugbank`

In [ ]:
drugBankDrugData_approved_ADMET_summary = pd.read_csv(dataDir + 'Toxicity/DrugBank/drugBankDrugData_ADMET_summary.csv')
drugBankDrugData_approved_ADMET_summary

### Filter compounds based on the results from Drugbank's `ADMET` summary

In [ ]:
# ── Build range lookup dict from summary dataframe ─────────────────────────────
# Ensure Endpoint is the index and numeric types are correct
summaryDF = drugBankDrugData_approved_ADMET_summary.copy()

if "Endpoint" in summaryDF.columns:
    summaryDF = summaryDF.set_index("Endpoint")

summaryDF = summaryDF.astype(float)

rangeDict = summaryDF[["Min", "Max"]].to_dict(orient="index")

# ── Range mask builder ─────────────────────────────────────────────────────────
def buildRangeMask(df, endpointCols, rangeDict):
    """
    Returns a boolean Series: True if ALL available endpoints in endpointCols
    fall within [Min, Max] defined in rangeDict.
    Skips endpoints missing from either the dataframe or the rangeDict.
    """
    mask = pd.Series(True, index=df.index)
    skipped = []
    applied = []

    for col in endpointCols:
        if col not in df.columns or col not in rangeDict:
            skipped.append(col)
            continue
        colMin = rangeDict[col]["Min"]
        colMax = rangeDict[col]["Max"]
        mask = mask & df[col].between(colMin, colMax, inclusive="both")
        applied.append(col)

    return mask, applied, skipped


df = BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity.copy()
total = len(df)

# ── Apply masks per category ───────────────────────────────────────────────────
coreMask,        coreApplied,        coreSkipped        = buildRangeMask(df, coreToxicityCols,     rangeDict)
mechanisticMask, mechanisticApplied, mechanisticSkipped = buildRangeMask(df, mechanisticAlertCols, rangeDict)
admeMask,        admeApplied,        admeSkipped        = buildRangeMask(df, admeLiabilityCols,    rangeDict)

# Combined: must pass all three categories simultaneously
combinedMask = coreMask & mechanisticMask & admeMask

# ── Filter dataframes ──────────────────────────────────────────────────────────
df_coreFiltered        = df[coreMask].reset_index(drop=True)
df_mechanisticFiltered = df[mechanisticMask].reset_index(drop=True)
df_admeFiltered        = df[admeMask].reset_index(drop=True)
df_allFiltered         = df[combinedMask].reset_index(drop=True)

# ── Report 
print("         DrugBank Approved Range Filtering Summary")
print(f"Total input molecules: {total}")

print(f"\n──> Core Toxicity Endpoints ")
print(f"  Endpoints applied : {coreApplied}")
print(f"  Endpoints skipped : {coreSkipped}")
print(f"  Passing molecules : {len(df_coreFiltered)} / {total}  ({len(df_coreFiltered)/total*100:.2f}%)")

print(f"\n──> Mechanistic Alert Endpoints ")
print(f"  Endpoints applied : {mechanisticApplied}")
print(f"  Endpoints skipped : {mechanisticSkipped}")
print(f"  Passing molecules : {len(df_mechanisticFiltered)} / {total}  ({len(df_mechanisticFiltered)/total*100:.2f}%)")

print(f"\n──> ADME / Metabolism Liability Endpoints ")
print(f"  Endpoints applied : {admeApplied}")
print(f"  Endpoints skipped : {admeSkipped}")
print(f"  Passing molecules : {len(df_admeFiltered)} / {total}  ({len(df_admeFiltered)/total*100:.2f}%)")

print(f"\n──> Combined (ALL categories must pass) ")
print(f"  Passing molecules : {len(df_allFiltered)} / {total}  ({len(df_allFiltered)/total*100:.2f}%)")
print("=" * 65)

# ── Inspect results ────────────────────────────────────────────────────────────
print("\nCombined filtered dataframe shape:", df_allFiltered.shape)
BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity = df_allFiltered.copy()
BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity

### Multiparameter optimization (MPO) framework based ADMET scoring

We replaced simple arithmetic averaging with a weighted desirability-based multiparameter optimization (MPO) framework. Each ADMET endpoint was converted to a 0–1 desirability value, where `1 --> favorable` and `0 --> unfavorable` profile. Category-level scores were then combined using weighted geometric means, following the same general pattern used in QED and ADMET-score approaches.

**References**

1. ADMET-score – a comprehensive scoring function for evaluation of chemical drug-likeness (https://pubs.rsc.org/en/content/articlelanding/2019/md/c8md00472b)
2. Quantifying the chemical beauty of drugs (https://www.nature.com/articles/nchem.1243)

In [ ]:
df = BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity.copy()


# Endpoint groups
coreToxicityCols = [
    "AMES",
    "hERG",
    "DILI",
    "ClinTox",
    "Carcinogens_Lagunin",
    "Skin_Reaction",
    "LD50_Zhu",
]

mechanisticAlertCols = [
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
    "NR-AR-LBD",
    "NR-AR",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER-LBD",
    "NR-ER",
    "NR-PPAR-gamma",
]

# Define these lists according to how you want to treat directionality
# lower-is-better => toxicity/liability style probabilities
admeLowerIsBetter = [
    "Pgp_Broccatelli",
    "CYP1A2_Veith",
    "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels",
    "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels",
    "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels",
    "CYP3A4_Veith",
]

# higher-is-better => absorption / permeability / bioavailability style endpoints
admeHigherIsBetter = [
    "Caco2_Wang",
    "HIA_Hou",
    "PAMPA_NCATS",
    "Bioavailability_Ma",
]

# Optional / context-dependent endpoint
# For non-CNS antivirals, you may wish to penalize BBB penetration.
# For CNS programs, you may want the opposite or exclude it from the score.
includeBBB = False
if includeBBB and "BBB_Martins" in df.columns:
    admeLowerIsBetter = admeLowerIsBetter + ["BBB_Martins"]


# Utilities
epsilon = 1e-6

def clip01(series):
    return series.clip(lower=epsilon, upper=1 - epsilon)

def desirabilityFromRiskProbability(series):
    """
    For toxicity/liability probabilities:
    lower risk is better -> desirability = 1 - p
    """
    return clip01(1 - series)

def desirabilityFromBenefitProbability(series):
    """
    For favorable probabilities/scores already in [0,1]:
    higher is better -> desirability = p
    """
    return clip01(series)

def desirabilityFromLD50(series, lowerQuantile=0.10, upperQuantile=0.90):
    """
    LD50_Zhu is treated as a regression-style acute toxicity endpoint.
    Higher LD50_Zhu implies greater acute toxicity risk.
    Map lower values -> high desirability, higher values -> low desirability.

    Uses a robust percentile-based linear desirability transform so the score
    is less sensitive to extreme outliers than plain min-max scaling.
    """
    qLow = series.quantile(lowerQuantile)
    qHigh = series.quantile(upperQuantile)

    if pd.isna(qLow) or pd.isna(qHigh) or qHigh == qLow:
        return pd.Series(0.5, index=series.index)

    # lower LD50_Zhu = better, higher = worse
    desirability = 1 - ((series - qLow) / (qHigh - qLow))
    return clip01(desirability)

def weightedGeometricMean(dfValues, weightsDict):
    """
    Compute weighted geometric mean across available columns only.
    """
    availableCols = [col for col in weightsDict if col in dfValues.columns]
    if len(availableCols) == 0:
        return pd.Series(np.nan, index=dfValues.index)

    weights = np.array([weightsDict[col] for col in availableCols], dtype=float)
    weights = weights / weights.sum()

    values = dfValues[availableCols].copy()
    values = values.clip(lower=epsilon, upper=1 - epsilon)

    logWeighted = np.log(values).mul(weights, axis=1).sum(axis=1)
    return np.exp(logWeighted)

# -----------------------------------------------------------------------------
# 1. Core toxicity desirability
# -----------------------------------------------------------------------------
coreWeights = {
    "AMES": 1.0,
    "hERG": 1.5,
    "DILI": 1.5,
    "ClinTox": 1.5,
    "Carcinogens_Lagunin": 1.0,
    "Skin_Reaction": 0.5,
    "LD50_Zhu": 1.25,
}

coreDesirabilityDF = pd.DataFrame(index=df.index)

for colName in coreToxicityCols:
    if colName not in df.columns:
        continue

    if colName == "LD50_Zhu":
        coreDesirabilityDF[colName] = desirabilityFromLD50(df[colName])
    else:
        coreDesirabilityDF[colName] = desirabilityFromRiskProbability(df[colName])

df["Core_Toxicity_Desirability"] = weightedGeometricMean(coreDesirabilityDF, coreWeights)

# Optional inverse form if you want "higher = worse"
df["Core_Toxicity_Score"] = 1 - df["Core_Toxicity_Desirability"]

# -----------------------------------------------------------------------------
# 2. Mechanistic alert desirability
# -----------------------------------------------------------------------------
mechWeights = {
    "SR-ARE": 1.0,
    "SR-ATAD5": 1.0,
    "SR-HSE": 0.75,
    "SR-MMP": 1.25,
    "SR-p53": 1.0,
    "NR-AR-LBD": 0.75,
    "NR-AR": 0.75,
    "NR-AhR": 0.75,
    "NR-Aromatase": 0.75,
    "NR-ER-LBD": 0.75,
    "NR-ER": 0.75,
    "NR-PPAR-gamma": 0.5,
}

mechDesirabilityDF = pd.DataFrame(index=df.index)
for colName in mechanisticAlertCols:
    if colName in df.columns:
        mechDesirabilityDF[colName] = desirabilityFromRiskProbability(df[colName])

df["Mechanistic_Alert_Desirability"] = weightedGeometricMean(mechDesirabilityDF, mechWeights)
df["Mechanistic_Alert_Score"] = 1 - df["Mechanistic_Alert_Desirability"]

# -----------------------------------------------------------------------------
# 3. ADME liability desirability
# -----------------------------------------------------------------------------
admeWeights = {
    "BBB_Martins": 0.5,   # context dependent
    "Pgp_Broccatelli": 1.0,
    "Caco2_Wang": 1.0,
    "HIA_Hou": 1.0,
    "PAMPA_NCATS": 0.75,
    "Bioavailability_Ma": 1.25,
    "CYP1A2_Veith": 1.0,
    "CYP2C19_Veith": 1.0,
    "CYP2C9_Substrate_CarbonMangels": 0.75,
    "CYP2C9_Veith": 1.0,
    "CYP2D6_Substrate_CarbonMangels": 0.75,
    "CYP2D6_Veith": 1.0,
    "CYP3A4_Substrate_CarbonMangels": 0.75,
    "CYP3A4_Veith": 1.0,
}

admeDesirabilityDF = pd.DataFrame(index=df.index)

for colName in admeLowerIsBetter:
    if colName in df.columns:
        admeDesirabilityDF[colName] = desirabilityFromRiskProbability(df[colName])

for colName in admeHigherIsBetter:
    if colName in df.columns:
        admeDesirabilityDF[colName] = desirabilityFromBenefitProbability(df[colName])

df["ADME_Desirability"] = weightedGeometricMean(admeDesirabilityDF, admeWeights)
df["ADME_Liability_Score"] = 1 - df["ADME_Desirability"]

# -----------------------------------------------------------------------------
# 4. Final MPO-style safety score
# -----------------------------------------------------------------------------
# Higher desirability = better
# We give core toxicity the highest influence, then ADME, then mechanistic alerts.
componentDesirabilityDF = pd.DataFrame({
    "Core_Toxicity_Desirability": df["Core_Toxicity_Desirability"],
    "Mechanistic_Alert_Desirability": df["Mechanistic_Alert_Desirability"],
    "ADME_Desirability": df["ADME_Desirability"],
})

componentWeights = {
    "Core_Toxicity_Desirability": 0.50,
    "Mechanistic_Alert_Desirability": 0.20,
    "ADME_Desirability": 0.30,
}

df["Safety_MPO_Desirability"] = weightedGeometricMean(
    componentDesirabilityDF,
    componentWeights
)

# Optional inverse form if you want higher = worse
df["Safety_MPO_RiskScore"] = 1 - df["Safety_MPO_Desirability"]

# -----------------------------------------------------------------------------
# 5. Markdown-style explanation column
# -----------------------------------------------------------------------------
markdownNote = (
    "Composite scores were computed using a weighted desirability-based MPO framework. "
    "Each endpoint was transformed to a 0–1 desirability value, then combined with a "
    "weighted geometric mean. Core toxicity used AMES, hERG, DILI, ClinTox, "
    "Carcinogens_Lagunin, Skin_Reaction, and LD50_Zhu; mechanistic score used SR/NR "
    "alerts; ADME score used direction-adjusted absorption, transporter, and CYP endpoints. "
    "This follows the general scoring pattern used in QED and ADMET-score literature."
)

df["Scoring_Method_Markdown"] = markdownNote

# -----------------------------------------------------------------------------
# 6. Final output
# -----------------------------------------------------------------------------
BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted = df[
    [
        "SMILES",
        "pPotency_prediction",
        "pPotency_std",
        "Core_Toxicity_Desirability",
        "Core_Toxicity_Score",
        "Mechanistic_Alert_Desirability",
        "Mechanistic_Alert_Score",
        "ADME_Desirability",
        "ADME_Liability_Score",
        "Safety_MPO_Desirability",
        "Safety_MPO_RiskScore",
        "Scoring_Method_Markdown",
    ]
].copy()

BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted

### Define filter for compound ranking based on `ADMET` scores

In [ ]:
pPotency_prediction_cutoff = 5
Core_Toxicity_cutoff = 0.3
Mechanistic_Alert_cutoff = 0.3
ADME_Liability_cutoff = 0.3

In [ ]:
BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_core = (
    BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted[
        (BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted["pPotency_prediction"] > pPotency_prediction_cutoff) &
        (BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted["Core_Toxicity_Score"] < Core_Toxicity_cutoff)
    ]
    .reset_index(drop=True)
)

BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_core
BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_core.to_csv(resultsDir + 
                                                                           'BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_nonToxic_core.csv', index=False, encoding="utf-8")

### Plot full distribution + highlighted filtered dataframe

In [ ]:
plotDF = BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted.copy()
highlightDF = BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_core.copy()

# Rebuild mask from predefined cutoffs
highlightMask = (
    (plotDF["pPotency_prediction"] > pPotency_prediction_cutoff) &
    (plotDF["Core_Toxicity_Score"] < Core_Toxicity_cutoff)
)

backgroundDF = plotDF[~highlightMask]

otherCount = len(backgroundDF)
highlightCount = len(highlightDF)

plt.figure(figsize=(9, 6))

# Plot all other compounds
plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["Core_Toxicity_Score"],
    alpha=0.45,
    marker=".",
    c="royalblue",
    label=f"Other compounds (n={otherCount})"
)

# Plot filtered core compounds
plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["Core_Toxicity_Score"],
    alpha=0.5,
    marker="*",
    s=140,
    c="g",
    linewidths=1.8,
    label=f"pPotency > {pPotency_prediction_cutoff} and Core_Toxicity_Score < {Core_Toxicity_cutoff} (n={highlightCount})"
)

plt.axvline(
    x=pPotency_prediction_cutoff,
    c="r",
    linestyle="--",
    linewidth=1,
    label=f"pPotency cutoff = {pPotency_prediction_cutoff}"
)

plt.axhline(
    y=Core_Toxicity_cutoff,
    c="r",
    linestyle="--",
    linewidth=1,
    label=f"Core toxicity cutoff = {Core_Toxicity_cutoff}"
)

plt.xlabel("Predicted pPotency")
plt.ylabel("Core Toxicity Score")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=2)
plt.tight_layout()
plt.show()

### Create filtered dataframe with `Mechanistic_Alert_Score` & `ADME_Liability_Score`

In [ ]:
# Select best overall compounds
BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_allEndpoints = (
    BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted[
        (BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted["pPotency_prediction"] >= pPotency_prediction_cutoff) &
        (BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted["Core_Toxicity_Score"] < Core_Toxicity_cutoff) &
        (BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted["Mechanistic_Alert_Score"] < Mechanistic_Alert_cutoff) &
        (BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted["ADME_Liability_Score"] < ADME_Liability_cutoff)
    ]
    .copy()
)


BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_allEndpoints.to_csv(resultsDir + 
                                                                           'BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_allEndpoints.csv', index=False, encoding="utf-8")
BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_allEndpoints

In [ ]:
plotDF = BacillusAnthracisVirus_wART_BuyableDataset_predicted_all_wToxicity_sorted.copy()

highlightMask = (
    (plotDF["pPotency_prediction"] >= pPotency_prediction_cutoff) &
    (plotDF["Core_Toxicity_Score"] < Core_Toxicity_cutoff) &
    (plotDF["Mechanistic_Alert_Score"] < Mechanistic_Alert_cutoff) &
    (plotDF["ADME_Liability_Score"] < ADME_Liability_cutoff)
)

backgroundDF = plotDF[~highlightMask]
highlightDF = plotDF[highlightMask].copy()

otherCount = len(backgroundDF)
highlightCount = len(highlightDF)
totalCount = len(plotDF)

print("Total compounds:", totalCount)
print("Best overall compounds:", highlightCount)

plt.figure(figsize=(10, 6))

# Full distributions
plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["Core_Toxicity_Score"],
    alpha=0.45,
    marker=".",
    c="royalblue",
    label=f"Core_Toxicity_Score: other compounds (n={otherCount})"
)

plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["Mechanistic_Alert_Score"],
    alpha=0.45,
    marker=".",
    c="darkorange",
    label=f"Mechanistic_Alert_Score: other compounds (n={otherCount})"
)

plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["ADME_Liability_Score"],
    alpha=0.45,
    marker=".",
    c="seagreen",
    label=f"ADME_Liability_Score: other compounds (n={otherCount})"
)

# Highlight best compounds
plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["Core_Toxicity_Score"],
    alpha=1.0,
    marker="*",
    s=180,
    c="navy",
    edgecolors="black",
    linewidths=0.8,
    label=f"Best compounds: core toxicity (n={highlightCount})"
)

plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["Mechanistic_Alert_Score"],
    alpha=1.0,
    marker="*",
    s=180,
    c="firebrick",
    edgecolors="black",
    linewidths=0.8,
    label=f"Best compounds: mechanistic concern (n={highlightCount})"
)

plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["ADME_Liability_Score"],
    alpha=1.0,
    marker="*",
    s=180,
    c="darkgreen",
    edgecolors="black",
    linewidths=0.8,
    label=f"Best compounds: ADME liability (n={highlightCount})"
)

plt.axvline(
    x=pPotency_prediction_cutoff,
    c="black",
    linestyle="--",
    linewidth=1.2,
    label=f"pPotency cutoff = {pPotency_prediction_cutoff}"
)

plt.axhline(
    y=Core_Toxicity_cutoff,
    c="navy",
    linestyle="--",
    linewidth=1.2,
    label=f"Core toxicity cutoff = {Core_Toxicity_cutoff}"
)

plt.axhline(
    y=Mechanistic_Alert_cutoff,
    c="firebrick",
    linestyle="--",
    linewidth=1.2,
    label=f"Mechanistic cutoff = {Mechanistic_Alert_cutoff}"
)

plt.axhline(
    y=ADME_Liability_cutoff,
    c="darkgreen",
    linestyle=":",
    linewidth=1.2,
    label=f"ADME cutoff = {ADME_Liability_cutoff}"
)

plt.xlabel("Predicted pPotency")
plt.ylabel("Score")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.32), ncol=2)
plt.tight_layout()
plt.show()

## Check toxicty for `Inosibe + Adenosine + Guanosine + Xanthosine` DORANET generated AntiviralsData set

In [ ]:
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all = pd.read_csv(ARTresultsDir + 'BacillusAnthracis_allDORAnetGenerated_Antivirals_ARTprediction.csv')
max_val = BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all['pPotency_prediction'].max()
min_val = BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all['pPotency_prediction'].min()
print(f"pPotency_prediction range: {min_val:.3f} → {max_val:.3f}")
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all

### Read predicted ADMET properties for all candidate SMILES

In [ ]:
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity = pd.read_csv(ARTresultsDir + 'BacillusAnthracis_allDORAnetGenerated_Antivirals_ARTprediction_wToxicity.csv')
max_val = BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity['pPotency_prediction'].max()
min_val = BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity['pPotency_prediction'].min()
print(f"pPotency_prediction range: {min_val:.3f} → {max_val:.3f}")
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity

### Retain potency predictions along with toxicity, mechanistic, and ADME-related endpoints

In [ ]:
# Potency-related columns
potencyCols = [
    "SMILES",
    "pPotency_prediction", "pPotency_std",
    "pPotency_lower_95CI", "pPotency_upper_95CI",
    "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI"
]

# Core toxicity endpoints
coreToxicityCols = [
    "AMES",
    "hERG",
    "DILI",
    "ClinTox",
    "Carcinogens_Lagunin",
    "Skin_Reaction",
    "LD50_Zhu"
]

# Mechanistic alert endpoints
mechanisticAlertCols = [
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
    "NR-AR-LBD",
    "NR-AR",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER-LBD",
    "NR-ER",
    "NR-PPAR-gamma"
]

# Mechanistic alert endpoints
# ADME / metabolism liability endpoints
admeLiabilityCols = [
    "BBB_Martins",
    "Pgp_Broccatelli",
    "Caco2_Wang",
    "HIA_Hou",
    "PAMPA_NCATS",
    "Bioavailability_Ma",
    "CYP1A2_Veith",
    "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels",
    "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels",
    "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels",
    "CYP3A4_Veith"
]

# Keep only columns that exist
selectedCols = potencyCols + [
    col for col in (coreToxicityCols + mechanisticAlertCols + admeLiabilityCols)
    if col in BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity.columns
]

BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity = BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity[selectedCols].copy()

print("Filtered dataframe shape:", BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity.shape)
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity.head()

### Filter compounds based on the results from Drugbank's `ADMET` summary

In [ ]:
# ── Build range lookup dict from summary dataframe ─────────────────────────────
# Ensure Endpoint is the index and numeric types are correct
summaryDF = drugBankDrugData_approved_ADMET_summary.copy()

if "Endpoint" in summaryDF.columns:
    summaryDF = summaryDF.set_index("Endpoint")

summaryDF = summaryDF.astype(float)

rangeDict = summaryDF[["Min", "Max"]].to_dict(orient="index")

# ── Range mask builder ─────────────────────────────────────────────────────────
def buildRangeMask(df, endpointCols, rangeDict):
    """
    Returns a boolean Series: True if ALL available endpoints in endpointCols
    fall within [Min, Max] defined in rangeDict.
    Skips endpoints missing from either the dataframe or the rangeDict.
    """
    mask = pd.Series(True, index=df.index)
    skipped = []
    applied = []

    for col in endpointCols:
        if col not in df.columns or col not in rangeDict:
            skipped.append(col)
            continue
        colMin = rangeDict[col]["Min"]
        colMax = rangeDict[col]["Max"]
        mask = mask & df[col].between(colMin, colMax, inclusive="both")
        applied.append(col)

    return mask, applied, skipped


df = BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity.copy()
total = len(df)

# ── Apply masks per category ───────────────────────────────────────────────────
coreMask,        coreApplied,        coreSkipped        = buildRangeMask(df, coreToxicityCols,     rangeDict)
mechanisticMask, mechanisticApplied, mechanisticSkipped = buildRangeMask(df, mechanisticAlertCols, rangeDict)
admeMask,        admeApplied,        admeSkipped        = buildRangeMask(df, admeLiabilityCols,    rangeDict)

# Combined: must pass all three categories simultaneously
combinedMask = coreMask & mechanisticMask & admeMask

# ── Filter dataframes ──────────────────────────────────────────────────────────
df_coreFiltered        = df[coreMask].reset_index(drop=True)
df_mechanisticFiltered = df[mechanisticMask].reset_index(drop=True)
df_admeFiltered        = df[admeMask].reset_index(drop=True)
df_allFiltered         = df[combinedMask].reset_index(drop=True)

# ── Report 
print("         DrugBank Approved Range Filtering Summary")
print(f"Total input molecules: {total}")

print(f"\n──> Core Toxicity Endpoints ")
print(f"  Endpoints applied : {coreApplied}")
print(f"  Endpoints skipped : {coreSkipped}")
print(f"  Passing molecules : {len(df_coreFiltered)} / {total}  ({len(df_coreFiltered)/total*100:.2f}%)")

print(f"\n──> Mechanistic Alert Endpoints ")
print(f"  Endpoints applied : {mechanisticApplied}")
print(f"  Endpoints skipped : {mechanisticSkipped}")
print(f"  Passing molecules : {len(df_mechanisticFiltered)} / {total}  ({len(df_mechanisticFiltered)/total*100:.2f}%)")

print(f"\n──> ADME / Metabolism Liability Endpoints ")
print(f"  Endpoints applied : {admeApplied}")
print(f"  Endpoints skipped : {admeSkipped}")
print(f"  Passing molecules : {len(df_admeFiltered)} / {total}  ({len(df_admeFiltered)/total*100:.2f}%)")

print(f"\n──> Combined (ALL categories must pass) ")
print(f"  Passing molecules : {len(df_allFiltered)} / {total}  ({len(df_allFiltered)/total*100:.2f}%)")
print("=" * 65)

# ── Inspect results ────────────────────────────────────────────────────────────
print("\nCombined filtered dataframe shape:", df_allFiltered.shape)
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity = df_allFiltered.copy()
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity

### Multiparameter optimization (MPO) framework based ADMET scoring

We replaced simple arithmetic averaging with a weighted desirability-based multiparameter optimization (MPO) framework. Each ADMET endpoint was converted to a 0–1 desirability value, where `1 --> favorable` and `0 --> unfavorable` profile. Category-level scores were then combined using weighted geometric means, following the same general pattern used in QED and ADMET-score approaches.

**References**

1. ADMET-score – a comprehensive scoring function for evaluation of chemical drug-likeness (https://pubs.rsc.org/en/content/articlelanding/2019/md/c8md00472b)
2. Quantifying the chemical beauty of drugs (https://www.nature.com/articles/nchem.1243)

In [ ]:
df = BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity.copy()


# Endpoint groups
coreToxicityCols = [
    "AMES",
    "hERG",
    "DILI",
    "ClinTox",
    "Carcinogens_Lagunin",
    "Skin_Reaction",
    "LD50_Zhu",
]

mechanisticAlertCols = [
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
    "NR-AR-LBD",
    "NR-AR",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER-LBD",
    "NR-ER",
    "NR-PPAR-gamma",
]

# Define these lists according to how you want to treat directionality
# lower-is-better => toxicity/liability style probabilities
admeLowerIsBetter = [
    "Pgp_Broccatelli",
    "CYP1A2_Veith",
    "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels",
    "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels",
    "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels",
    "CYP3A4_Veith",
]

# higher-is-better => absorption / permeability / bioavailability style endpoints
admeHigherIsBetter = [
    "Caco2_Wang",
    "HIA_Hou",
    "PAMPA_NCATS",
    "Bioavailability_Ma",
]

# Optional / context-dependent endpoint
# For non-CNS antivirals, you may wish to penalize BBB penetration.
# For CNS programs, you may want the opposite or exclude it from the score.
includeBBB = False
if includeBBB and "BBB_Martins" in df.columns:
    admeLowerIsBetter = admeLowerIsBetter + ["BBB_Martins"]


# Utilities
epsilon = 1e-6

def clip01(series):
    return series.clip(lower=epsilon, upper=1 - epsilon)

def desirabilityFromRiskProbability(series):
    """
    For toxicity/liability probabilities:
    lower risk is better -> desirability = 1 - p
    """
    return clip01(1 - series)

def desirabilityFromBenefitProbability(series):
    """
    For favorable probabilities/scores already in [0,1]:
    higher is better -> desirability = p
    """
    return clip01(series)

def desirabilityFromLD50(series, lowerQuantile=0.10, upperQuantile=0.90):
    """
    LD50_Zhu is treated as a regression-style acute toxicity endpoint.
    Higher LD50_Zhu implies greater acute toxicity risk.
    Map lower values -> high desirability, higher values -> low desirability.

    Uses a robust percentile-based linear desirability transform so the score
    is less sensitive to extreme outliers than plain min-max scaling.
    """
    qLow = series.quantile(lowerQuantile)
    qHigh = series.quantile(upperQuantile)

    if pd.isna(qLow) or pd.isna(qHigh) or qHigh == qLow:
        return pd.Series(0.5, index=series.index)

    # lower LD50_Zhu = better, higher = worse
    desirability = 1 - ((series - qLow) / (qHigh - qLow))
    return clip01(desirability)

def weightedGeometricMean(dfValues, weightsDict):
    """
    Compute weighted geometric mean across available columns only.
    """
    availableCols = [col for col in weightsDict if col in dfValues.columns]
    if len(availableCols) == 0:
        return pd.Series(np.nan, index=dfValues.index)

    weights = np.array([weightsDict[col] for col in availableCols], dtype=float)
    weights = weights / weights.sum()

    values = dfValues[availableCols].copy()
    values = values.clip(lower=epsilon, upper=1 - epsilon)

    logWeighted = np.log(values).mul(weights, axis=1).sum(axis=1)
    return np.exp(logWeighted)

# -----------------------------------------------------------------------------
# 1. Core toxicity desirability
# -----------------------------------------------------------------------------
coreWeights = {
    "AMES": 1.0,
    "hERG": 1.5,
    "DILI": 1.5,
    "ClinTox": 1.5,
    "Carcinogens_Lagunin": 1.0,
    "Skin_Reaction": 0.5,
    "LD50_Zhu": 1.25,
}

coreDesirabilityDF = pd.DataFrame(index=df.index)

for colName in coreToxicityCols:
    if colName not in df.columns:
        continue

    if colName == "LD50_Zhu":
        coreDesirabilityDF[colName] = desirabilityFromLD50(df[colName])
    else:
        coreDesirabilityDF[colName] = desirabilityFromRiskProbability(df[colName])

df["Core_Toxicity_Desirability"] = weightedGeometricMean(coreDesirabilityDF, coreWeights)

# Optional inverse form if you want "higher = worse"
df["Core_Toxicity_Score"] = 1 - df["Core_Toxicity_Desirability"]

# -----------------------------------------------------------------------------
# 2. Mechanistic alert desirability
# -----------------------------------------------------------------------------
mechWeights = {
    "SR-ARE": 1.0,
    "SR-ATAD5": 1.0,
    "SR-HSE": 0.75,
    "SR-MMP": 1.25,
    "SR-p53": 1.0,
    "NR-AR-LBD": 0.75,
    "NR-AR": 0.75,
    "NR-AhR": 0.75,
    "NR-Aromatase": 0.75,
    "NR-ER-LBD": 0.75,
    "NR-ER": 0.75,
    "NR-PPAR-gamma": 0.5,
}

mechDesirabilityDF = pd.DataFrame(index=df.index)
for colName in mechanisticAlertCols:
    if colName in df.columns:
        mechDesirabilityDF[colName] = desirabilityFromRiskProbability(df[colName])

df["Mechanistic_Alert_Desirability"] = weightedGeometricMean(mechDesirabilityDF, mechWeights)
df["Mechanistic_Alert_Score"] = 1 - df["Mechanistic_Alert_Desirability"]

# -----------------------------------------------------------------------------
# 3. ADME liability desirability
# -----------------------------------------------------------------------------
admeWeights = {
    "BBB_Martins": 0.5,   # context dependent
    "Pgp_Broccatelli": 1.0,
    "Caco2_Wang": 1.0,
    "HIA_Hou": 1.0,
    "PAMPA_NCATS": 0.75,
    "Bioavailability_Ma": 1.25,
    "CYP1A2_Veith": 1.0,
    "CYP2C19_Veith": 1.0,
    "CYP2C9_Substrate_CarbonMangels": 0.75,
    "CYP2C9_Veith": 1.0,
    "CYP2D6_Substrate_CarbonMangels": 0.75,
    "CYP2D6_Veith": 1.0,
    "CYP3A4_Substrate_CarbonMangels": 0.75,
    "CYP3A4_Veith": 1.0,
}

admeDesirabilityDF = pd.DataFrame(index=df.index)

for colName in admeLowerIsBetter:
    if colName in df.columns:
        admeDesirabilityDF[colName] = desirabilityFromRiskProbability(df[colName])

for colName in admeHigherIsBetter:
    if colName in df.columns:
        admeDesirabilityDF[colName] = desirabilityFromBenefitProbability(df[colName])

df["ADME_Desirability"] = weightedGeometricMean(admeDesirabilityDF, admeWeights)
df["ADME_Liability_Score"] = 1 - df["ADME_Desirability"]

# -----------------------------------------------------------------------------
# 4. Final MPO-style safety score
# -----------------------------------------------------------------------------
# Higher desirability = better
# We give core toxicity the highest influence, then ADME, then mechanistic alerts.
componentDesirabilityDF = pd.DataFrame({
    "Core_Toxicity_Desirability": df["Core_Toxicity_Desirability"],
    "Mechanistic_Alert_Desirability": df["Mechanistic_Alert_Desirability"],
    "ADME_Desirability": df["ADME_Desirability"],
})

componentWeights = {
    "Core_Toxicity_Desirability": 0.50,
    "Mechanistic_Alert_Desirability": 0.20,
    "ADME_Desirability": 0.30,
}

df["Safety_MPO_Desirability"] = weightedGeometricMean(
    componentDesirabilityDF,
    componentWeights
)

# Optional inverse form if you want higher = worse
df["Safety_MPO_RiskScore"] = 1 - df["Safety_MPO_Desirability"]

# -----------------------------------------------------------------------------
# 5. Markdown-style explanation column
# -----------------------------------------------------------------------------
markdownNote = (
    "Composite scores were computed using a weighted desirability-based MPO framework. "
    "Each endpoint was transformed to a 0–1 desirability value, then combined with a "
    "weighted geometric mean. Core toxicity used AMES, hERG, DILI, ClinTox, "
    "Carcinogens_Lagunin, Skin_Reaction, and LD50_Zhu; mechanistic score used SR/NR "
    "alerts; ADME score used direction-adjusted absorption, transporter, and CYP endpoints. "
    "This follows the general scoring pattern used in QED and ADMET-score literature."
)

df["Scoring_Method_Markdown"] = markdownNote

# -----------------------------------------------------------------------------
# 6. Final output
# -----------------------------------------------------------------------------
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted = df[
    [
        "SMILES",
        "pPotency_prediction",
        "pPotency_std",
        "Core_Toxicity_Desirability",
        "Core_Toxicity_Score",
        "Mechanistic_Alert_Desirability",
        "Mechanistic_Alert_Score",
        "ADME_Desirability",
        "ADME_Liability_Score",
        "Safety_MPO_Desirability",
        "Safety_MPO_RiskScore",
        "Scoring_Method_Markdown",
    ]
].copy()

BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted

### Define filter for compound ranking based on `ADMET` scores

In [ ]:
pPotency_prediction_cutoff = 5
Core_Toxicity_cutoff = 0.3
Mechanistic_Alert_cutoff = 0.3
ADME_Liability_cutoff = 0.3

In [ ]:
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_core = (
    BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted[
        (BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted["pPotency_prediction"] > pPotency_prediction_cutoff) &
        (BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted["Core_Toxicity_Score"] < Core_Toxicity_cutoff)
    ]
    .reset_index(drop=True)
)


BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_core.to_csv(resultsDir + 
                                                                           'BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_nonToxic_core.csv', index=False, encoding="utf-8")
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_core

### Plot full distribution + highlighted filtered dataframe

In [ ]:
plotDF = BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted.copy()
highlightDF = BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_core.copy()

# Rebuild mask from predefined cutoffs
highlightMask = (
    (plotDF["pPotency_prediction"] > pPotency_prediction_cutoff) &
    (plotDF["Core_Toxicity_Score"] < Core_Toxicity_cutoff)
)

backgroundDF = plotDF[~highlightMask]

otherCount = len(backgroundDF)
highlightCount = len(highlightDF)

plt.figure(figsize=(9, 6))

# Plot all other compounds
plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["Core_Toxicity_Score"],
    alpha=0.45,
    marker=".",
    c="royalblue",
    label=f"Other compounds (n={otherCount})"
)

# Plot filtered core compounds
plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["Core_Toxicity_Score"],
    alpha=0.5,
    marker="*",
    s=140,
    c="g",
    linewidths=1.8,
    label=f"pPotency > {pPotency_prediction_cutoff} and Core_Toxicity_Score < {Core_Toxicity_cutoff} (n={highlightCount})"
)

plt.axvline(
    x=pPotency_prediction_cutoff,
    c="r",
    linestyle="--",
    linewidth=1,
    label=f"pPotency cutoff = {pPotency_prediction_cutoff}"
)

plt.axhline(
    y=Core_Toxicity_cutoff,
    c="r",
    linestyle="--",
    linewidth=1,
    label=f"Core toxicity cutoff = {Core_Toxicity_cutoff}"
)

plt.xlabel("Predicted pPotency")
plt.ylabel("Core Toxicity Score")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=2)
plt.tight_layout()
plt.show()

### Create filtered dataframe with `Mechanistic_Alert_Score` & `ADME_Liability_Score`

In [ ]:
# Select best overall compounds
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_allEndpoints = (
    BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted[
        (BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted["pPotency_prediction"] >= pPotency_prediction_cutoff) &
        (BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted["Core_Toxicity_Score"] < Core_Toxicity_cutoff) &
        (BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted["Mechanistic_Alert_Score"] < Mechanistic_Alert_cutoff) &
        (BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted["ADME_Liability_Score"] < ADME_Liability_cutoff)
    ]
    .copy()
)


BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_allEndpoints.to_csv(resultsDir + 
                                                                           'BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_allEndpoints.csv', index=False, encoding="utf-8")
BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_allEndpoints

In [ ]:
plotDF = BacillusAnthracisVirus_wART_allDORAnetGenerated_predicted_all_wToxicity_sorted.copy()

highlightMask = (
    (plotDF["pPotency_prediction"] >= pPotency_prediction_cutoff) &
    (plotDF["Core_Toxicity_Score"] < Core_Toxicity_cutoff) &
    (plotDF["Mechanistic_Alert_Score"] < Mechanistic_Alert_cutoff) &
    (plotDF["ADME_Liability_Score"] < ADME_Liability_cutoff)
)

backgroundDF = plotDF[~highlightMask]
highlightDF = plotDF[highlightMask].copy()

otherCount = len(backgroundDF)
highlightCount = len(highlightDF)
totalCount = len(plotDF)

print("Total compounds:", totalCount)
print("Best overall compounds:", highlightCount)

plt.figure(figsize=(10, 6))

# Full distributions
plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["Core_Toxicity_Score"],
    alpha=0.45,
    marker=".",
    c="royalblue",
    label=f"Core_Toxicity_Score: other compounds (n={otherCount})"
)

plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["Mechanistic_Alert_Score"],
    alpha=0.45,
    marker=".",
    c="darkorange",
    label=f"Mechanistic_Alert_Score: other compounds (n={otherCount})"
)

plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["ADME_Liability_Score"],
    alpha=0.45,
    marker=".",
    c="seagreen",
    label=f"ADME_Liability_Score: other compounds (n={otherCount})"
)

# Highlight best compounds
plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["Core_Toxicity_Score"],
    alpha=1.0,
    marker="*",
    s=180,
    c="navy",
    edgecolors="black",
    linewidths=0.8,
    label=f"Best compounds: core toxicity (n={highlightCount})"
)

plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["Mechanistic_Alert_Score"],
    alpha=1.0,
    marker="*",
    s=180,
    c="firebrick",
    edgecolors="black",
    linewidths=0.8,
    label=f"Best compounds: mechanistic concern (n={highlightCount})"
)

plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["ADME_Liability_Score"],
    alpha=1.0,
    marker="*",
    s=180,
    c="darkgreen",
    edgecolors="black",
    linewidths=0.8,
    label=f"Best compounds: ADME liability (n={highlightCount})"
)

plt.axvline(
    x=pPotency_prediction_cutoff,
    c="black",
    linestyle="--",
    linewidth=1.2,
    label=f"pPotency cutoff = {pPotency_prediction_cutoff}"
)

plt.axhline(
    y=Core_Toxicity_cutoff,
    c="navy",
    linestyle="--",
    linewidth=1.2,
    label=f"Core toxicity cutoff = {Core_Toxicity_cutoff}"
)

plt.axhline(
    y=Mechanistic_Alert_cutoff,
    c="firebrick",
    linestyle="--",
    linewidth=1.2,
    label=f"Mechanistic cutoff = {Mechanistic_Alert_cutoff}"
)

plt.axhline(
    y=ADME_Liability_cutoff,
    c="darkgreen",
    linestyle=":",
    linewidth=1.2,
    label=f"ADME cutoff = {ADME_Liability_cutoff}"
)

plt.xlabel("Predicted pPotency")
plt.ylabel("Score")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.32), ncol=2)
plt.tight_layout()
plt.show()

## Check toxicty for `Inosine-substructures` DORANET generated AntiviralsData set

In [ ]:
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all = pd.read_csv(ARTresultsDir + 'BacillusAnthracisVirusData_InosineSubstructures_DORAnetGenerated_ARTprediction.csv')
max_val = BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all['pPotency_prediction'].max()
min_val = BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all['pPotency_prediction'].min()
print(f"pPotency_prediction range: {min_val:.3f} → {max_val:.3f}")
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all

### Read predicted ADMET properties for all candidate SMILES

In [ ]:
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity = pd.read_csv(ARTresultsDir + 'BacillusAnthracisVirusData_InosineSubstructures_DORAnetGenerated_ARTprediction_wToxicity.csv')
max_val = BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity['pPotency_prediction'].max()
min_val = BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity['pPotency_prediction'].min()
print(f"pPotency_prediction range: {min_val:.3f} → {max_val:.3f}")
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity

### Retain potency predictions along with toxicity, mechanistic, and ADME-related endpoints

In [ ]:
# Potency-related columns
potencyCols = [
    "SMILES",
    "pPotency_prediction", "pPotency_std",
    "pPotency_lower_95CI", "pPotency_upper_95CI",
    "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI"
]

# Core toxicity endpoints
coreToxicityCols = [
    "AMES",
    "hERG",
    "DILI",
    "ClinTox",
    "Carcinogens_Lagunin",
    "Skin_Reaction",
    "LD50_Zhu"
]

# Mechanistic alert endpoints
mechanisticAlertCols = [
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
    "NR-AR-LBD",
    "NR-AR",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER-LBD",
    "NR-ER",
    "NR-PPAR-gamma"
]

# Mechanistic alert endpoints
# ADME / metabolism liability endpoints
admeLiabilityCols = [
    "BBB_Martins",
    "Pgp_Broccatelli",
    "Caco2_Wang",
    "HIA_Hou",
    "PAMPA_NCATS",
    "Bioavailability_Ma",
    "CYP1A2_Veith",
    "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels",
    "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels",
    "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels",
    "CYP3A4_Veith"
]

# Keep only columns that exist
selectedCols = potencyCols + [
    col for col in (coreToxicityCols + mechanisticAlertCols + admeLiabilityCols)
    if col in BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity.columns
]

BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity = BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity[selectedCols].copy()

print("Filtered dataframe shape:", BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity.shape)
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity.head()

### Filter compounds based on the results from Drugbank's `ADMET` summary

In [ ]:
# ── Build range lookup dict from summary dataframe ─────────────────────────────
# Ensure Endpoint is the index and numeric types are correct
summaryDF = drugBankDrugData_approved_ADMET_summary.copy()

if "Endpoint" in summaryDF.columns:
    summaryDF = summaryDF.set_index("Endpoint")

summaryDF = summaryDF.astype(float)

rangeDict = summaryDF[["Min", "Max"]].to_dict(orient="index")

# ── Range mask builder ─────────────────────────────────────────────────────────
def buildRangeMask(df, endpointCols, rangeDict):
    """
    Returns a boolean Series: True if ALL available endpoints in endpointCols
    fall within [Min, Max] defined in rangeDict.
    Skips endpoints missing from either the dataframe or the rangeDict.
    """
    mask = pd.Series(True, index=df.index)
    skipped = []
    applied = []

    for col in endpointCols:
        if col not in df.columns or col not in rangeDict:
            skipped.append(col)
            continue
        colMin = rangeDict[col]["Min"]
        colMax = rangeDict[col]["Max"]
        mask = mask & df[col].between(colMin, colMax, inclusive="both")
        applied.append(col)

    return mask, applied, skipped


df = BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity.copy()
total = len(df)

# ── Apply masks per category ───────────────────────────────────────────────────
coreMask,        coreApplied,        coreSkipped        = buildRangeMask(df, coreToxicityCols,     rangeDict)
mechanisticMask, mechanisticApplied, mechanisticSkipped = buildRangeMask(df, mechanisticAlertCols, rangeDict)
admeMask,        admeApplied,        admeSkipped        = buildRangeMask(df, admeLiabilityCols,    rangeDict)

# Combined: must pass all three categories simultaneously
combinedMask = coreMask & mechanisticMask & admeMask

# ── Filter dataframes ──────────────────────────────────────────────────────────
df_coreFiltered        = df[coreMask].reset_index(drop=True)
df_mechanisticFiltered = df[mechanisticMask].reset_index(drop=True)
df_admeFiltered        = df[admeMask].reset_index(drop=True)
df_allFiltered         = df[combinedMask].reset_index(drop=True)

# ── Report 
print("         DrugBank Approved Range Filtering Summary")
print(f"Total input molecules: {total}")

print(f"\n──> Core Toxicity Endpoints ")
print(f"  Endpoints applied : {coreApplied}")
print(f"  Endpoints skipped : {coreSkipped}")
print(f"  Passing molecules : {len(df_coreFiltered)} / {total}  ({len(df_coreFiltered)/total*100:.2f}%)")

print(f"\n──> Mechanistic Alert Endpoints ")
print(f"  Endpoints applied : {mechanisticApplied}")
print(f"  Endpoints skipped : {mechanisticSkipped}")
print(f"  Passing molecules : {len(df_mechanisticFiltered)} / {total}  ({len(df_mechanisticFiltered)/total*100:.2f}%)")

print(f"\n──> ADME / Metabolism Liability Endpoints ")
print(f"  Endpoints applied : {admeApplied}")
print(f"  Endpoints skipped : {admeSkipped}")
print(f"  Passing molecules : {len(df_admeFiltered)} / {total}  ({len(df_admeFiltered)/total*100:.2f}%)")

print(f"\n──> Combined (ALL categories must pass) ")
print(f"  Passing molecules : {len(df_allFiltered)} / {total}  ({len(df_allFiltered)/total*100:.2f}%)")
print("=" * 65)

# ── Inspect results ────────────────────────────────────────────────────────────
print("\nCombined filtered dataframe shape:", df_allFiltered.shape)
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity = df_allFiltered.copy()
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity

### Multiparameter optimization (MPO) framework based ADMET scoring

We replaced simple arithmetic averaging with a weighted desirability-based multiparameter optimization (MPO) framework. Each ADMET endpoint was converted to a 0–1 desirability value, where `1 --> favorable` and `0 --> unfavorable` profile. Category-level scores were then combined using weighted geometric means, following the same general pattern used in QED and ADMET-score approaches.

**References**

1. ADMET-score – a comprehensive scoring function for evaluation of chemical drug-likeness (https://pubs.rsc.org/en/content/articlelanding/2019/md/c8md00472b)
2. Quantifying the chemical beauty of drugs (https://www.nature.com/articles/nchem.1243)

In [ ]:
df = BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity.copy()


# Endpoint groups
coreToxicityCols = [
    "AMES",
    "hERG",
    "DILI",
    "ClinTox",
    "Carcinogens_Lagunin",
    "Skin_Reaction",
    "LD50_Zhu",
]

mechanisticAlertCols = [
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
    "NR-AR-LBD",
    "NR-AR",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER-LBD",
    "NR-ER",
    "NR-PPAR-gamma",
]

# Define these lists according to how you want to treat directionality
# lower-is-better => toxicity/liability style probabilities
admeLowerIsBetter = [
    "Pgp_Broccatelli",
    "CYP1A2_Veith",
    "CYP2C19_Veith",
    "CYP2C9_Substrate_CarbonMangels",
    "CYP2C9_Veith",
    "CYP2D6_Substrate_CarbonMangels",
    "CYP2D6_Veith",
    "CYP3A4_Substrate_CarbonMangels",
    "CYP3A4_Veith",
]

# higher-is-better => absorption / permeability / bioavailability style endpoints
admeHigherIsBetter = [
    "Caco2_Wang",
    "HIA_Hou",
    "PAMPA_NCATS",
    "Bioavailability_Ma",
]

# Optional / context-dependent endpoint
# For non-CNS antivirals, you may wish to penalize BBB penetration.
# For CNS programs, you may want the opposite or exclude it from the score.
includeBBB = False
if includeBBB and "BBB_Martins" in df.columns:
    admeLowerIsBetter = admeLowerIsBetter + ["BBB_Martins"]


# Utilities
epsilon = 1e-6

def clip01(series):
    return series.clip(lower=epsilon, upper=1 - epsilon)

def desirabilityFromRiskProbability(series):
    """
    For toxicity/liability probabilities:
    lower risk is better -> desirability = 1 - p
    """
    return clip01(1 - series)

def desirabilityFromBenefitProbability(series):
    """
    For favorable probabilities/scores already in [0,1]:
    higher is better -> desirability = p
    """
    return clip01(series)

def desirabilityFromLD50(series, lowerQuantile=0.10, upperQuantile=0.90):
    """
    LD50_Zhu is treated as a regression-style acute toxicity endpoint.
    Higher LD50_Zhu implies greater acute toxicity risk.
    Map lower values -> high desirability, higher values -> low desirability.

    Uses a robust percentile-based linear desirability transform so the score
    is less sensitive to extreme outliers than plain min-max scaling.
    """
    qLow = series.quantile(lowerQuantile)
    qHigh = series.quantile(upperQuantile)

    if pd.isna(qLow) or pd.isna(qHigh) or qHigh == qLow:
        return pd.Series(0.5, index=series.index)

    # lower LD50_Zhu = better, higher = worse
    desirability = 1 - ((series - qLow) / (qHigh - qLow))
    return clip01(desirability)

def weightedGeometricMean(dfValues, weightsDict):
    """
    Compute weighted geometric mean across available columns only.
    """
    availableCols = [col for col in weightsDict if col in dfValues.columns]
    if len(availableCols) == 0:
        return pd.Series(np.nan, index=dfValues.index)

    weights = np.array([weightsDict[col] for col in availableCols], dtype=float)
    weights = weights / weights.sum()

    values = dfValues[availableCols].copy()
    values = values.clip(lower=epsilon, upper=1 - epsilon)

    logWeighted = np.log(values).mul(weights, axis=1).sum(axis=1)
    return np.exp(logWeighted)

# -----------------------------------------------------------------------------
# 1. Core toxicity desirability
# -----------------------------------------------------------------------------
coreWeights = {
    "AMES": 1.0,
    "hERG": 1.5,
    "DILI": 1.5,
    "ClinTox": 1.5,
    "Carcinogens_Lagunin": 1.0,
    "Skin_Reaction": 0.5,
    "LD50_Zhu": 1.25,
}

coreDesirabilityDF = pd.DataFrame(index=df.index)

for colName in coreToxicityCols:
    if colName not in df.columns:
        continue

    if colName == "LD50_Zhu":
        coreDesirabilityDF[colName] = desirabilityFromLD50(df[colName])
    else:
        coreDesirabilityDF[colName] = desirabilityFromRiskProbability(df[colName])

df["Core_Toxicity_Desirability"] = weightedGeometricMean(coreDesirabilityDF, coreWeights)

# Optional inverse form if you want "higher = worse"
df["Core_Toxicity_Score"] = 1 - df["Core_Toxicity_Desirability"]

# -----------------------------------------------------------------------------
# 2. Mechanistic alert desirability
# -----------------------------------------------------------------------------
mechWeights = {
    "SR-ARE": 1.0,
    "SR-ATAD5": 1.0,
    "SR-HSE": 0.75,
    "SR-MMP": 1.25,
    "SR-p53": 1.0,
    "NR-AR-LBD": 0.75,
    "NR-AR": 0.75,
    "NR-AhR": 0.75,
    "NR-Aromatase": 0.75,
    "NR-ER-LBD": 0.75,
    "NR-ER": 0.75,
    "NR-PPAR-gamma": 0.5,
}

mechDesirabilityDF = pd.DataFrame(index=df.index)
for colName in mechanisticAlertCols:
    if colName in df.columns:
        mechDesirabilityDF[colName] = desirabilityFromRiskProbability(df[colName])

df["Mechanistic_Alert_Desirability"] = weightedGeometricMean(mechDesirabilityDF, mechWeights)
df["Mechanistic_Alert_Score"] = 1 - df["Mechanistic_Alert_Desirability"]

# -----------------------------------------------------------------------------
# 3. ADME liability desirability
# -----------------------------------------------------------------------------
admeWeights = {
    "BBB_Martins": 0.5,   # context dependent
    "Pgp_Broccatelli": 1.0,
    "Caco2_Wang": 1.0,
    "HIA_Hou": 1.0,
    "PAMPA_NCATS": 0.75,
    "Bioavailability_Ma": 1.25,
    "CYP1A2_Veith": 1.0,
    "CYP2C19_Veith": 1.0,
    "CYP2C9_Substrate_CarbonMangels": 0.75,
    "CYP2C9_Veith": 1.0,
    "CYP2D6_Substrate_CarbonMangels": 0.75,
    "CYP2D6_Veith": 1.0,
    "CYP3A4_Substrate_CarbonMangels": 0.75,
    "CYP3A4_Veith": 1.0,
}

admeDesirabilityDF = pd.DataFrame(index=df.index)

for colName in admeLowerIsBetter:
    if colName in df.columns:
        admeDesirabilityDF[colName] = desirabilityFromRiskProbability(df[colName])

for colName in admeHigherIsBetter:
    if colName in df.columns:
        admeDesirabilityDF[colName] = desirabilityFromBenefitProbability(df[colName])

df["ADME_Desirability"] = weightedGeometricMean(admeDesirabilityDF, admeWeights)
df["ADME_Liability_Score"] = 1 - df["ADME_Desirability"]

# -----------------------------------------------------------------------------
# 4. Final MPO-style safety score
# -----------------------------------------------------------------------------
# Higher desirability = better
# We give core toxicity the highest influence, then ADME, then mechanistic alerts.
componentDesirabilityDF = pd.DataFrame({
    "Core_Toxicity_Desirability": df["Core_Toxicity_Desirability"],
    "Mechanistic_Alert_Desirability": df["Mechanistic_Alert_Desirability"],
    "ADME_Desirability": df["ADME_Desirability"],
})

componentWeights = {
    "Core_Toxicity_Desirability": 0.50,
    "Mechanistic_Alert_Desirability": 0.20,
    "ADME_Desirability": 0.30,
}

df["Safety_MPO_Desirability"] = weightedGeometricMean(
    componentDesirabilityDF,
    componentWeights
)

# Optional inverse form if you want higher = worse
df["Safety_MPO_RiskScore"] = 1 - df["Safety_MPO_Desirability"]

# -----------------------------------------------------------------------------
# 5. Markdown-style explanation column
# -----------------------------------------------------------------------------
markdownNote = (
    "Composite scores were computed using a weighted desirability-based MPO framework. "
    "Each endpoint was transformed to a 0–1 desirability value, then combined with a "
    "weighted geometric mean. Core toxicity used AMES, hERG, DILI, ClinTox, "
    "Carcinogens_Lagunin, Skin_Reaction, and LD50_Zhu; mechanistic score used SR/NR "
    "alerts; ADME score used direction-adjusted absorption, transporter, and CYP endpoints. "
    "This follows the general scoring pattern used in QED and ADMET-score literature."
)

df["Scoring_Method_Markdown"] = markdownNote

# -----------------------------------------------------------------------------
# 6. Final output
# -----------------------------------------------------------------------------
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted = df[
    [
        "SMILES",
        "pPotency_prediction",
        "pPotency_std",
        "Core_Toxicity_Desirability",
        "Core_Toxicity_Score",
        "Mechanistic_Alert_Desirability",
        "Mechanistic_Alert_Score",
        "ADME_Desirability",
        "ADME_Liability_Score",
        "Safety_MPO_Desirability",
        "Safety_MPO_RiskScore",
        "Scoring_Method_Markdown",
    ]
].copy()

BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted

### Define filter for compound ranking based on `ADMET` scores

In [ ]:
pPotency_prediction_cutoff = 5
Core_Toxicity_cutoff = 0.3
Mechanistic_Alert_cutoff = 0.3
ADME_Liability_cutoff = 0.3

In [ ]:
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_core = (
    BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted[
        (BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted["pPotency_prediction"] > pPotency_prediction_cutoff) &
        (BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted["Core_Toxicity_Score"] < Core_Toxicity_cutoff)
    ]
    .reset_index(drop=True)
)


BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_core.to_csv(resultsDir + 
                                                                           'BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_nonToxic_core.csv', index=False, encoding="utf-8")
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_core

### Plot full distribution + highlighted filtered dataframe

In [ ]:
plotDF = BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted.copy()
highlightDF = BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_core.copy()

# Rebuild mask from predefined cutoffs
highlightMask = (
    (plotDF["pPotency_prediction"] > pPotency_prediction_cutoff) &
    (plotDF["Core_Toxicity_Score"] < Core_Toxicity_cutoff)
)

backgroundDF = plotDF[~highlightMask]

otherCount = len(backgroundDF)
highlightCount = len(highlightDF)

plt.figure(figsize=(9, 6))

# Plot all other compounds
plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["Core_Toxicity_Score"],
    alpha=0.45,
    marker=".",
    c="royalblue",
    label=f"Other compounds (n={otherCount})"
)

# Plot filtered core compounds
plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["Core_Toxicity_Score"],
    alpha=0.5,
    marker="*",
    s=140,
    c="g",
    linewidths=1.8,
    label=f"pPotency > {pPotency_prediction_cutoff} and Core_Toxicity_Score < {Core_Toxicity_cutoff} (n={highlightCount})"
)

plt.axvline(
    x=pPotency_prediction_cutoff,
    c="r",
    linestyle="--",
    linewidth=1,
    label=f"pPotency cutoff = {pPotency_prediction_cutoff}"
)

plt.axhline(
    y=Core_Toxicity_cutoff,
    c="r",
    linestyle="--",
    linewidth=1,
    label=f"Core toxicity cutoff = {Core_Toxicity_cutoff}"
)

plt.xlabel("Predicted pPotency")
plt.ylabel("Core Toxicity Score")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=2)
plt.tight_layout()
plt.show()

### Create filtered dataframe with `Mechanistic_Alert_Score` & `ADME_Liability_Score`

In [ ]:
# Select best overall compounds
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_allEndpoints = (
    BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted[
        (BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted["pPotency_prediction"] >= pPotency_prediction_cutoff) &
        (BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted["Core_Toxicity_Score"] < Core_Toxicity_cutoff) &
        (BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted["Mechanistic_Alert_Score"] < Mechanistic_Alert_cutoff) &
        (BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted["ADME_Liability_Score"] < ADME_Liability_cutoff)
    ]
    .copy()
)


BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_allEndpoints.to_csv(resultsDir + 
                                                                           'BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_allEndpoints.csv', index=False, encoding="utf-8")
BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_allEndpoints

In [ ]:
plotDF = BacillusAnthracisVirus_wART_InosineSubstructures_predicted_all_wToxicity_sorted.copy()

highlightMask = (
    (plotDF["pPotency_prediction"] >= pPotency_prediction_cutoff) &
    (plotDF["Core_Toxicity_Score"] < Core_Toxicity_cutoff) &
    (plotDF["Mechanistic_Alert_Score"] < Mechanistic_Alert_cutoff) &
    (plotDF["ADME_Liability_Score"] < ADME_Liability_cutoff)
)

backgroundDF = plotDF[~highlightMask]
highlightDF = plotDF[highlightMask].copy()

otherCount = len(backgroundDF)
highlightCount = len(highlightDF)
totalCount = len(plotDF)

print("Total compounds:", totalCount)
print("Best overall compounds:", highlightCount)

plt.figure(figsize=(10, 6))

# Full distributions
plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["Core_Toxicity_Score"],
    alpha=0.45,
    marker=".",
    c="royalblue",
    label=f"Core_Toxicity_Score: other compounds (n={otherCount})"
)

plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["Mechanistic_Alert_Score"],
    alpha=0.45,
    marker=".",
    c="darkorange",
    label=f"Mechanistic_Alert_Score: other compounds (n={otherCount})"
)

plt.scatter(
    backgroundDF["pPotency_prediction"],
    backgroundDF["ADME_Liability_Score"],
    alpha=0.45,
    marker=".",
    c="seagreen",
    label=f"ADME_Liability_Score: other compounds (n={otherCount})"
)

# Highlight best compounds
plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["Core_Toxicity_Score"],
    alpha=1.0,
    marker="*",
    s=180,
    c="navy",
    edgecolors="black",
    linewidths=0.8,
    label=f"Best compounds: core toxicity (n={highlightCount})"
)

plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["Mechanistic_Alert_Score"],
    alpha=1.0,
    marker="*",
    s=180,
    c="firebrick",
    edgecolors="black",
    linewidths=0.8,
    label=f"Best compounds: mechanistic concern (n={highlightCount})"
)

plt.scatter(
    highlightDF["pPotency_prediction"],
    highlightDF["ADME_Liability_Score"],
    alpha=1.0,
    marker="*",
    s=180,
    c="darkgreen",
    edgecolors="black",
    linewidths=0.8,
    label=f"Best compounds: ADME liability (n={highlightCount})"
)

plt.axvline(
    x=pPotency_prediction_cutoff,
    c="black",
    linestyle="--",
    linewidth=1.2,
    label=f"pPotency cutoff = {pPotency_prediction_cutoff}"
)

plt.axhline(
    y=Core_Toxicity_cutoff,
    c="navy",
    linestyle="--",
    linewidth=1.2,
    label=f"Core toxicity cutoff = {Core_Toxicity_cutoff}"
)

plt.axhline(
    y=Mechanistic_Alert_cutoff,
    c="firebrick",
    linestyle="--",
    linewidth=1.2,
    label=f"Mechanistic cutoff = {Mechanistic_Alert_cutoff}"
)

plt.axhline(
    y=ADME_Liability_cutoff,
    c="darkgreen",
    linestyle=":",
    linewidth=1.2,
    label=f"ADME cutoff = {ADME_Liability_cutoff}"
)

plt.xlabel("Predicted pPotency")
plt.ylabel("Score")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.32), ncol=2)
plt.tight_layout()
plt.show()